# EpiSentinel — Data Transformation Pipeline
**Purpose:** Transform Synthea-generated data into an Africa-context master dataset for epidemic early warning system training.

**Sources:**
- Synthea (patient records generator)
- WHO AFRO disease distribution data
- Africa CDC priority disease list 2024
- WHO disease fact sheets (symptoms)

**Output:** `master_dataset.csv` — ready for AI model training

---

## Step 1: Install & Import Libraries

In [2]:
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

BASE_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\csv'

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 2: Load Raw Synthea Data

In [2]:
print('Loading data — this may take a few minutes for large files...')

patients = pd.read_csv(f'{BASE_PATH}\\patients.csv')

conditions = pd.read_csv(f'{BASE_PATH}\\conditions.csv')

# Load only needed columns from encounters to save memory
encounters = pd.read_csv(
    f'{BASE_PATH}\\encounters.csv',
    usecols=['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'ENCOUNTERCLASS']
)

organizations = pd.read_csv(f'{BASE_PATH}\\organizations.csv')

print(f'Patients     : {len(patients):,}')
print(f'Conditions   : {len(conditions):,}')
print(f'Encounters   : {len(encounters):,}')
print(f'Organizations: {len(organizations):,}')

Loading data — this may take a few minutes for large files...
Patients     : 114,685
Conditions   : 4,164,676
Encounters   : 6,744,416
Organizations: 1,235


## Step 3: Define African Cities with Population Weights
**Source:** UN World Population Prospects + national census data  
Higher weight = more patients assigned to that city (reflects real population density)

In [3]:
african_cities = [
    # Egypt
    {'CITY': 'Cairo',           'STATE': 'Cairo Governorate',           'COUNTRY': 'Egypt',        'LAT': 30.0444,  'LON': 31.2357,  'WEIGHT': 10},
    {'CITY': 'Alexandria',      'STATE': 'Alexandria Governorate',      'COUNTRY': 'Egypt',        'LAT': 31.2001,  'LON': 29.9187,  'WEIGHT': 6},
    {'CITY': 'Giza',            'STATE': 'Giza Governorate',            'COUNTRY': 'Egypt',        'LAT': 30.0131,  'LON': 31.2089,  'WEIGHT': 5},
    {'CITY': 'Shubra El Kheima','STATE': 'Qalyubia Governorate',        'COUNTRY': 'Egypt',        'LAT': 30.1286,  'LON': 31.2422,  'WEIGHT': 3},
    {'CITY': 'Mansoura',        'STATE': 'Dakahlia Governorate',        'COUNTRY': 'Egypt',        'LAT': 31.0409,  'LON': 31.3785,  'WEIGHT': 2},
    {'CITY': 'Tanta',           'STATE': 'Gharbia Governorate',         'COUNTRY': 'Egypt',        'LAT': 30.7865,  'LON': 31.0004,  'WEIGHT': 2},
    {'CITY': 'Asyut',           'STATE': 'Asyut Governorate',           'COUNTRY': 'Egypt',        'LAT': 27.1809,  'LON': 31.1837,  'WEIGHT': 2},
    {'CITY': 'Minya',           'STATE': 'Minya Governorate',           'COUNTRY': 'Egypt',        'LAT': 28.0871,  'LON': 30.7618,  'WEIGHT': 2},
    {'CITY': 'Port Said',       'STATE': 'Port Said Governorate',       'COUNTRY': 'Egypt',        'LAT': 31.2653,  'LON': 32.3019,  'WEIGHT': 1},
    {'CITY': 'Suez',            'STATE': 'Suez Governorate',            'COUNTRY': 'Egypt',        'LAT': 29.9668,  'LON': 32.5498,  'WEIGHT': 1},
    {'CITY': 'Luxor',           'STATE': 'Luxor Governorate',           'COUNTRY': 'Egypt',        'LAT': 25.6872,  'LON': 32.6396,  'WEIGHT': 1},
    {'CITY': 'Aswan',           'STATE': 'Aswan Governorate',           'COUNTRY': 'Egypt',        'LAT': 24.0889,  'LON': 32.8998,  'WEIGHT': 1},
    {'CITY': 'Ismailia',        'STATE': 'Ismailia Governorate',        'COUNTRY': 'Egypt',        'LAT': 30.5965,  'LON': 32.2715,  'WEIGHT': 1},
    {'CITY': 'Fayyum',          'STATE': 'Fayyum Governorate',          'COUNTRY': 'Egypt',        'LAT': 29.3084,  'LON': 30.8428,  'WEIGHT': 1},
    {'CITY': 'Zagazig',         'STATE': 'Sharqia Governorate',         'COUNTRY': 'Egypt',        'LAT': 30.5877,  'LON': 31.5021,  'WEIGHT': 1},
    {'CITY': 'Damietta',        'STATE': 'Damietta Governorate',        'COUNTRY': 'Egypt',        'LAT': 31.4165,  'LON': 31.8133,  'WEIGHT': 1},
    {'CITY': 'Beni Suef',       'STATE': 'Beni Suef Governorate',       'COUNTRY': 'Egypt',        'LAT': 29.0661,  'LON': 31.0994,  'WEIGHT': 1},
    {'CITY': 'Sohag',           'STATE': 'Sohag Governorate',           'COUNTRY': 'Egypt',        'LAT': 26.5590,  'LON': 31.6957,  'WEIGHT': 1},
    {'CITY': 'Qena',            'STATE': 'Qena Governorate',            'COUNTRY': 'Egypt',        'LAT': 26.1551,  'LON': 32.7160,  'WEIGHT': 1},
    {'CITY': 'Kafr El Sheikh',  'STATE': 'Kafr El Sheikh Governorate',  'COUNTRY': 'Egypt',        'LAT': 31.1107,  'LON': 30.9388,  'WEIGHT': 1},
    {'CITY': 'Damanhur',        'STATE': 'Beheira Governorate',         'COUNTRY': 'Egypt',        'LAT': 31.0341,  'LON': 30.4685,  'WEIGHT': 1},
    {'CITY': 'Shibin El Kom',   'STATE': 'Monufia Governorate',         'COUNTRY': 'Egypt',        'LAT': 30.5500,  'LON': 31.0100,  'WEIGHT': 1},
    {'CITY': 'Banha',           'STATE': 'Qalyubia Governorate',        'COUNTRY': 'Egypt',        'LAT': 30.4667,  'LON': 31.1833,  'WEIGHT': 1},
    {'CITY': 'Hurghada',        'STATE': 'Red Sea Governorate',         'COUNTRY': 'Egypt',        'LAT': 27.2579,  'LON': 33.8116,  'WEIGHT': 1},
    {'CITY': 'Marsa Matruh',    'STATE': 'Matrouh Governorate',         'COUNTRY': 'Egypt',        'LAT': 31.3543,  'LON': 27.2373,  'WEIGHT': 1},
    {'CITY': 'Arish',           'STATE': 'North Sinai Governorate',     'COUNTRY': 'Egypt',        'LAT': 31.1319,  'LON': 33.7985,  'WEIGHT': 1},
    {'CITY': 'Idfu',            'STATE': 'Aswan Governorate',           'COUNTRY': 'Egypt',        'LAT': 24.9780,  'LON': 32.8760,  'WEIGHT': 1},
    # Nigeria
    {'CITY': 'Lagos',           'STATE': 'Lagos State',                 'COUNTRY': 'Nigeria',      'LAT':  6.5244,  'LON':  3.3792,  'WEIGHT': 8},
    {'CITY': 'Kano',            'STATE': 'Kano State',                  'COUNTRY': 'Nigeria',      'LAT': 12.0022,  'LON':  8.5920,  'WEIGHT': 4},
    {'CITY': 'Ibadan',          'STATE': 'Oyo State',                   'COUNTRY': 'Nigeria',      'LAT':  7.3775,  'LON':  3.9470,  'WEIGHT': 3},
    {'CITY': 'Abuja',           'STATE': 'FCT',                         'COUNTRY': 'Nigeria',      'LAT':  9.0765,  'LON':  7.3986,  'WEIGHT': 2},
    {'CITY': 'Port Harcourt',   'STATE': 'Rivers State',                'COUNTRY': 'Nigeria',      'LAT':  4.8156,  'LON':  7.0498,  'WEIGHT': 2},
    {'CITY': 'Kaduna',          'STATE': 'Kaduna State',                'COUNTRY': 'Nigeria',      'LAT': 10.5222,  'LON':  7.4383,  'WEIGHT': 2},
    # Ethiopia
    {'CITY': 'Addis Ababa',     'STATE': 'Addis Ababa',                 'COUNTRY': 'Ethiopia',     'LAT':  9.0320,  'LON': 38.7469,  'WEIGHT': 5},
    {'CITY': 'Dire Dawa',       'STATE': 'Dire Dawa',                   'COUNTRY': 'Ethiopia',     'LAT':  9.5931,  'LON': 41.8661,  'WEIGHT': 2},
    {'CITY': 'Mekelle',         'STATE': 'Tigray',                      'COUNTRY': 'Ethiopia',     'LAT': 13.4967,  'LON': 39.4753,  'WEIGHT': 1},
    {'CITY': 'Hawassa',         'STATE': 'Sidama',                      'COUNTRY': 'Ethiopia',     'LAT':  7.0621,  'LON': 38.4762,  'WEIGHT': 1},
    # Kenya
    {'CITY': 'Nairobi',         'STATE': 'Nairobi County',              'COUNTRY': 'Kenya',        'LAT': -1.2921,  'LON': 36.8219,  'WEIGHT': 5},
    {'CITY': 'Mombasa',         'STATE': 'Mombasa County',              'COUNTRY': 'Kenya',        'LAT': -4.0435,  'LON': 39.6682,  'WEIGHT': 2},
    {'CITY': 'Kisumu',          'STATE': 'Kisumu County',               'COUNTRY': 'Kenya',        'LAT': -0.1022,  'LON': 34.7617,  'WEIGHT': 1},
    # Tanzania
    {'CITY': 'Dar es Salaam',   'STATE': 'Dar es Salaam',               'COUNTRY': 'Tanzania',     'LAT': -6.7924,  'LON': 39.2083,  'WEIGHT': 4},
    {'CITY': 'Dodoma',          'STATE': 'Dodoma',                      'COUNTRY': 'Tanzania',     'LAT': -6.1722,  'LON': 35.7395,  'WEIGHT': 1},
    # South Africa
    {'CITY': 'Johannesburg',    'STATE': 'Gauteng',                     'COUNTRY': 'South Africa', 'LAT':-26.2041,  'LON': 28.0473,  'WEIGHT': 4},
    {'CITY': 'Cape Town',       'STATE': 'Western Cape',                'COUNTRY': 'South Africa', 'LAT':-33.9249,  'LON': 18.4241,  'WEIGHT': 3},
    {'CITY': 'Durban',          'STATE': 'KwaZulu-Natal',               'COUNTRY': 'South Africa', 'LAT':-29.8587,  'LON': 31.0218,  'WEIGHT': 2},
    # Ghana
    {'CITY': 'Accra',           'STATE': 'Greater Accra',               'COUNTRY': 'Ghana',        'LAT':  5.6037,  'LON': -0.1870,  'WEIGHT': 3},
    {'CITY': 'Kumasi',          'STATE': 'Ashanti',                     'COUNTRY': 'Ghana',        'LAT':  6.6885,  'LON': -1.6244,  'WEIGHT': 2},
    # Senegal
    {'CITY': 'Dakar',           'STATE': 'Dakar Region',                'COUNTRY': 'Senegal',      'LAT': 14.7167,  'LON':-17.4677,  'WEIGHT': 3},
    # DRC
    {'CITY': 'Kinshasa',        'STATE': 'Kinshasa',                    'COUNTRY': 'DRC',          'LAT': -4.4419,  'LON': 15.2663,  'WEIGHT': 5},
    {'CITY': 'Lubumbashi',      'STATE': 'Haut-Katanga',                'COUNTRY': 'DRC',          'LAT':-11.6609,  'LON': 27.4794,  'WEIGHT': 2},
    {'CITY': 'Goma',            'STATE': 'North Kivu',                  'COUNTRY': 'DRC',          'LAT': -1.6796,  'LON': 29.2285,  'WEIGHT': 1},
    # Sudan
    {'CITY': 'Khartoum',        'STATE': 'Khartoum State',              'COUNTRY': 'Sudan',        'LAT': 15.5007,  'LON': 32.5599,  'WEIGHT': 3},
    {'CITY': 'Omdurman',        'STATE': 'Khartoum State',              'COUNTRY': 'Sudan',        'LAT': 15.6445,  'LON': 32.4777,  'WEIGHT': 2},
    # Uganda
    {'CITY': 'Kampala',         'STATE': 'Kampala',                     'COUNTRY': 'Uganda',       'LAT':  0.3476,  'LON': 32.5825,  'WEIGHT': 3},
    # Zimbabwe
    {'CITY': 'Harare',          'STATE': 'Harare Province',             'COUNTRY': 'Zimbabwe',     'LAT':-17.8252,  'LON': 31.0335,  'WEIGHT': 2},
    # Morocco
    {'CITY': 'Casablanca',      'STATE': 'Casablanca-Settat',           'COUNTRY': 'Morocco',      'LAT': 33.5731,  'LON': -7.5898,  'WEIGHT': 3},
    {'CITY': 'Rabat',           'STATE': 'Rabat-Sale-Kenitra',          'COUNTRY': 'Morocco',      'LAT': 34.0209,  'LON': -6.8416,  'WEIGHT': 2},
    # Tunisia
    {'CITY': 'Tunis',           'STATE': 'Tunis Governorate',           'COUNTRY': 'Tunisia',      'LAT': 36.8190,  'LON': 10.1658,  'WEIGHT': 2},
    # Algeria
    {'CITY': 'Algiers',         'STATE': 'Algiers Province',            'COUNTRY': 'Algeria',      'LAT': 36.7372,  'LON':  3.0865,  'WEIGHT': 3},
    # Cameroon
    {'CITY': 'Yaounde',         'STATE': 'Centre Region',               'COUNTRY': 'Cameroon',     'LAT':  3.8480,  'LON': 11.5021,  'WEIGHT': 2},
    {'CITY': 'Douala',          'STATE': 'Littoral Region',             'COUNTRY': 'Cameroon',     'LAT':  4.0511,  'LON':  9.7679,  'WEIGHT': 2},
    # Mozambique
    {'CITY': 'Maputo',          'STATE': 'Maputo Province',             'COUNTRY': 'Mozambique',   'LAT':-25.9692,  'LON': 32.5732,  'WEIGHT': 2},
    # Zambia
    {'CITY': 'Lusaka',          'STATE': 'Lusaka Province',             'COUNTRY': 'Zambia',       'LAT':-15.4166,  'LON': 28.2833,  'WEIGHT': 2},
    # Mali
    {'CITY': 'Bamako',          'STATE': 'Bamako Capital District',     'COUNTRY': 'Mali',         'LAT': 12.6392,  'LON': -8.0029,  'WEIGHT': 2},
]

city_weights = np.array([c['WEIGHT'] for c in african_cities], dtype=float)
city_weights /= city_weights.sum()

print(f'Total cities : {len(african_cities)}')
print(f'Countries    : {len(set(c["COUNTRY"] for c in african_cities))}')

Total cities : 64
Countries    : 19


## Step 4: Define Priority Infectious Diseases & Symptoms
**Source:** WHO disease fact sheets + Africa CDC Priority Disease List 2024  
Symptoms are defined as binary columns (1 = present, 0 = absent) — best format for ML models

In [4]:
# All possible symptoms across all diseases
ALL_SYMPTOMS = [
    'sym_fever',
    'sym_chills',
    'sym_headache',
    'sym_fatigue',
    'sym_vomiting',
    'sym_diarrhea',
    'sym_cough',
    'sym_shortness_of_breath',
    'sym_chest_pain',
    'sym_rash',
    'sym_muscle_pain',
    'sym_abdominal_pain',
    'sym_night_sweats',
    'sym_weight_loss',
    'sym_jaundice',
    'sym_stiff_neck',
    'sym_sensitivity_to_light',
    'sym_red_eyes',
    'sym_runny_nose',
    'sym_sore_throat',
    'sym_hemorrhage',
    'sym_sweating',
    'sym_dehydration',
    'sym_muscle_cramps',
    'sym_loss_of_smell',
    'sym_swollen_lymph_nodes',
    'sym_pain_behind_eyes',
    'sym_general_weakness',
]

# Disease -> symptoms mapping
# Source: WHO fact sheets for each disease
# 1 = symptom always present, 0.7 = symptom often present, 0.4 = sometimes present
disease_symptom_probs = {
    'Malaria': {
        'sym_fever': 1.0, 'sym_chills': 0.9, 'sym_headache': 0.8,
        'sym_vomiting': 0.6, 'sym_fatigue': 0.7, 'sym_sweating': 0.8,
        'sym_muscle_pain': 0.5,
    },
    'Cholera': {
        'sym_diarrhea': 1.0, 'sym_vomiting': 0.8, 'sym_dehydration': 0.9,
        'sym_muscle_cramps': 0.6,
    },
    'Typhoid fever': {
        'sym_fever': 1.0, 'sym_headache': 0.8, 'sym_abdominal_pain': 0.7,
        'sym_fatigue': 0.7, 'sym_rash': 0.4,
    },
    'Tuberculosis': {
        'sym_cough': 1.0, 'sym_fever': 0.7, 'sym_night_sweats': 0.8,
        'sym_weight_loss': 0.8, 'sym_fatigue': 0.7, 'sym_chest_pain': 0.5,
    },
    'Ebola virus disease': {
        'sym_fever': 1.0, 'sym_headache': 0.9, 'sym_muscle_pain': 0.8,
        'sym_vomiting': 0.8, 'sym_diarrhea': 0.7, 'sym_hemorrhage': 0.6,
        'sym_fatigue': 0.8,
    },
    'Measles': {
        'sym_fever': 1.0, 'sym_cough': 0.9, 'sym_runny_nose': 0.8,
        'sym_red_eyes': 0.8, 'sym_rash': 1.0,
    },
    'Meningitis': {
        'sym_fever': 1.0, 'sym_headache': 1.0, 'sym_stiff_neck': 0.9,
        'sym_sensitivity_to_light': 0.7, 'sym_vomiting': 0.6,
    },
    'Dengue fever': {
        'sym_fever': 1.0, 'sym_headache': 0.9, 'sym_pain_behind_eyes': 0.8,
        'sym_muscle_pain': 0.8, 'sym_rash': 0.6, 'sym_fatigue': 0.7,
    },
    'Yellow fever': {
        'sym_fever': 1.0, 'sym_headache': 0.8, 'sym_jaundice': 0.7,
        'sym_muscle_pain': 0.7, 'sym_vomiting': 0.6, 'sym_fatigue': 0.7,
    },
    'Lassa fever': {
        'sym_fever': 1.0, 'sym_general_weakness': 0.9, 'sym_headache': 0.7,
        'sym_sore_throat': 0.7, 'sym_vomiting': 0.6, 'sym_chest_pain': 0.5,
    },
    'Rift Valley fever': {
        'sym_fever': 1.0, 'sym_headache': 0.8, 'sym_muscle_pain': 0.7,
        'sym_general_weakness': 0.7, 'sym_vomiting': 0.5,
    },
    'Mpox': {
        'sym_fever': 1.0, 'sym_rash': 1.0, 'sym_swollen_lymph_nodes': 0.9,
        'sym_headache': 0.7, 'sym_muscle_pain': 0.7, 'sym_fatigue': 0.7,
    },
    'COVID-19': {
        'sym_fever': 0.8, 'sym_cough': 0.9, 'sym_shortness_of_breath': 0.7,
        'sym_fatigue': 0.8, 'sym_loss_of_smell': 0.6, 'sym_headache': 0.6,
    },
    'Influenza': {
        'sym_fever': 0.9, 'sym_cough': 0.9, 'sym_sore_throat': 0.7,
        'sym_headache': 0.7, 'sym_fatigue': 0.8, 'sym_muscle_pain': 0.7,
    },
    'Pneumonia': {
        'sym_cough': 1.0, 'sym_fever': 0.8, 'sym_shortness_of_breath': 0.8,
        'sym_chest_pain': 0.6, 'sym_fatigue': 0.7,
    },
    'Diarrheal disease': {
        'sym_diarrhea': 1.0, 'sym_abdominal_pain': 0.8, 'sym_vomiting': 0.6,
        'sym_fever': 0.5, 'sym_dehydration': 0.7,
    },
    'Hepatitis A': {
        'sym_fever': 0.7, 'sym_fatigue': 0.8, 'sym_jaundice': 0.9,
        'sym_abdominal_pain': 0.7, 'sym_vomiting': 0.6,
    },
    'HIV/AIDS': {
        'sym_fever': 0.6, 'sym_fatigue': 0.9, 'sym_weight_loss': 0.8,
        'sym_night_sweats': 0.7, 'sym_swollen_lymph_nodes': 0.7,
    },
    'Leishmaniasis': {
        'sym_fever': 0.9, 'sym_weight_loss': 0.8, 'sym_fatigue': 0.8,
        'sym_general_weakness': 0.7,
    },
    'Schistosomiasis': {
        'sym_rash': 0.6, 'sym_fever': 0.7, 'sym_abdominal_pain': 0.7,
        'sym_diarrhea': 0.5, 'sym_fatigue': 0.6,
    },
}

def get_symptom_vector(disease):
    """Returns a binary symptom vector for a given disease.
    Each symptom is present (1) or absent (0) based on probability from WHO data."""
    probs = disease_symptom_probs.get(disease, {'sym_fever': 0.7, 'sym_fatigue': 0.6})
    vector = {}
    for sym in ALL_SYMPTOMS:
        prob = probs.get(sym, 0.05)  # 5% base chance for non-listed symptoms
        vector[sym] = 1 if random.random() < prob else 0
    return vector

print(f'Diseases defined  : {len(disease_symptom_probs)}')
print(f'Symptom columns   : {len(ALL_SYMPTOMS)}')
print('Sample symptom vector for Malaria:')
sample = get_symptom_vector('Malaria')
print({k: v for k, v in sample.items() if v == 1})

Diseases defined  : 20
Symptom columns   : 28
Sample symptom vector for Malaria:
{'sym_fever': 1, 'sym_chills': 1, 'sym_headache': 1, 'sym_fatigue': 1, 'sym_rash': 1, 'sym_muscle_pain': 1, 'sym_night_sweats': 1, 'sym_sore_throat': 1, 'sym_sweating': 1}


## Step 5: Define Disease Distribution per Country
**Source:** WHO AFRO epidemiological reports + Africa CDC 2024  
Rule-based: each country has different disease probabilities based on real epidemiological data

In [5]:
country_disease_distribution = {
    'Egypt': {
        'Typhoid fever': 0.20, 'Hepatitis A': 0.15, 'Influenza': 0.15,
        'COVID-19': 0.10, 'Malaria': 0.08, 'Tuberculosis': 0.08,
        'Meningitis': 0.05, 'Diarrheal disease': 0.07, 'Cholera': 0.05, 'Pneumonia': 0.07,
    },
    'Nigeria': {
        'Malaria': 0.30, 'Lassa fever': 0.10, 'Cholera': 0.12,
        'Typhoid fever': 0.10, 'Meningitis': 0.08, 'Tuberculosis': 0.08,
        'Measles': 0.08, 'COVID-19': 0.07, 'Pneumonia': 0.07,
    },
    'Ethiopia': {
        'Malaria': 0.25, 'Tuberculosis': 0.15, 'Cholera': 0.12,
        'Meningitis': 0.08, 'Dengue fever': 0.08, 'Leishmaniasis': 0.07,
        'Typhoid fever': 0.08, 'Measles': 0.08, 'Diarrheal disease': 0.09,
    },
    'Kenya': {
        'Malaria': 0.25, 'Cholera': 0.12, 'Tuberculosis': 0.12,
        'Dengue fever': 0.10, 'Typhoid fever': 0.10, 'Rift Valley fever': 0.06,
        'COVID-19': 0.08, 'Pneumonia': 0.09, 'Measles': 0.08,
    },
    'DRC': {
        'Malaria': 0.25, 'Ebola virus disease': 0.08, 'Cholera': 0.15,
        'Mpox': 0.08, 'Tuberculosis': 0.10, 'Measles': 0.10,
        'Typhoid fever': 0.08, 'Meningitis': 0.08, 'Pneumonia': 0.08,
    },
    'Uganda': {
        'Malaria': 0.28, 'Ebola virus disease': 0.05, 'Tuberculosis': 0.13,
        'Cholera': 0.10, 'Typhoid fever': 0.10, 'Measles': 0.10,
        'Meningitis': 0.08, 'Pneumonia': 0.08, 'Diarrheal disease': 0.08,
    },
    'Tanzania': {
        'Malaria': 0.28, 'Cholera': 0.12, 'Tuberculosis': 0.12,
        'Typhoid fever': 0.10, 'Dengue fever': 0.08, 'Measles': 0.08,
        'Pneumonia': 0.10, 'Diarrheal disease': 0.12,
    },
    'South Africa': {
        'Tuberculosis': 0.20, 'HIV/AIDS': 0.15, 'COVID-19': 0.15,
        'Influenza': 0.12, 'Cholera': 0.08, 'Pneumonia': 0.10,
        'Typhoid fever': 0.08, 'Measles': 0.07, 'Meningitis': 0.05,
    },
    'DEFAULT': {
        'Malaria': 0.20, 'Cholera': 0.12, 'Typhoid fever': 0.12,
        'Tuberculosis': 0.10, 'Measles': 0.10, 'Meningitis': 0.08,
        'Pneumonia': 0.10, 'Dengue fever': 0.08, 'COVID-19': 0.06, 'Diarrheal disease': 0.04,
    },
}

def assign_disease(country):
    dist = country_disease_distribution.get(country, country_disease_distribution['DEFAULT'])
    diseases = list(dist.keys())
    probs = np.array(list(dist.values()))
    probs /= probs.sum()  # normalize
    return np.random.choice(diseases, p=probs)

print('Country-disease distributions defined.')

Country-disease distributions defined.


## Step 6: Assign African Locations to Patients

In [6]:
print('Assigning African locations to patients...')

chosen_indices = np.random.choice(len(african_cities), size=len(patients), p=city_weights)
chosen_cities  = [african_cities[i] for i in chosen_indices]

patients['CITY']    = [c['CITY']    for c in chosen_cities]
patients['STATE']   = [c['STATE']   for c in chosen_cities]
patients['COUNTRY'] = [c['COUNTRY'] for c in chosen_cities]
patients['LAT']     = [c['LAT']     for c in chosen_cities]
patients['LON']     = [c['LON']     for c in chosen_cities]

patients_clean = patients[['Id', 'BIRTHDATE', 'DEATHDATE', 'GENDER',
                            'CITY', 'STATE', 'COUNTRY', 'LAT', 'LON']].copy()

print(f'Done. Patients: {len(patients_clean):,}')
print(patients_clean['COUNTRY'].value_counts().head(8))

Assigning African locations to patients...
Done. Patients: 114,685
COUNTRY
Egypt           39378
Nigeria         15962
Ethiopia         6990
South Africa     6931
DRC              6198
Kenya            6169
Morocco          3926
Sudan            3906
Name: count, dtype: int64


## Step 7: Process Encounters — Extract Temporal Features

In [7]:
print('Processing timestamps...')

# FIX: Synthea timestamps can mix timezone-aware and naive strings (e.g. some end in 'Z', some
# don't). Parsing them together with the default settings silently turns the mismatched ones
# into NaT (missing values) with NO warning, and this is also what caused the tz-naive vs
# tz-aware crash later at sort_values(). format='mixed' lets pandas infer each row's format
# individually; utc=True + tz_localize(None) then normalizes everything to one consistent
# (naive) timezone so nothing gets silently dropped or mismatched downstream.
encounters['VISIT_DATE'] = pd.to_datetime(
    encounters['START'], errors='coerce', format='mixed', utc=True
).dt.tz_localize(None)

n_lost = encounters['VISIT_DATE'].isna().sum()
if n_lost > 0:
    print(f'Warning: {n_lost:,} encounter rows had unparseable dates and were dropped.')

encounters['VISIT_YEAR']  = encounters['VISIT_DATE'].dt.year
encounters['VISIT_MONTH'] = encounters['VISIT_DATE'].dt.month
encounters['VISIT_WEEK']  = encounters['VISIT_DATE'].dt.isocalendar().week.astype('Int64')
encounters['VISIT_DAY']   = encounters['VISIT_DATE'].dt.day_of_year

# NOTE: SEASON now depends on COUNTRY (hemisphere fix below), so it's computed in Step 8
# after merging with patient location, not here.

# REPORTED_DATE: for real hospital records we don't have a separate reporting lag field
# in Synthea, so we treat VISIT_DATE as the reported date for the real data (the reporting
# delay simulation below only applies to the injected outbreak/noise records, where we
# control the ground truth and can model that lag realistically).
encounters['REPORTED_DATE'] = encounters['VISIT_DATE']

encounters_clean = encounters[['Id', 'PATIENT', 'ORGANIZATION',
                               'VISIT_DATE', 'REPORTED_DATE', 'VISIT_YEAR', 'VISIT_MONTH',
                               'VISIT_WEEK', 'VISIT_DAY',
                               'ENCOUNTERCLASS']].dropna(subset=['VISIT_DATE']).copy()

print(f'Done. Encounters: {len(encounters_clean):,}')


Processing timestamps...
Done. Encounters: 6,744,416


## Step 8: Merge Encounters + Patients + Assign Diseases

In [8]:
print('Merging data...')

df = encounters_clean.merge(
    patients_clean,
    left_on='PATIENT',
    right_on='Id',
    how='inner'
)

# --- Hemisphere-aware seasons ---
# The original SEASON logic assumed the same rainfall calendar for every country, but Southern
# Africa's rain/dry cycle runs about 6 months out of phase with the Sahel/East Africa pattern.
# Fixing this matters because SEASON is a genuine predictive feature (malaria/cholera peaks
# track the rains).
SOUTHERN_HEMISPHERE_COUNTRIES = {'South Africa', 'Zimbabwe', 'Mozambique', 'Zambia'}

def get_season(month, country):
    if country in SOUTHERN_HEMISPHERE_COUNTRIES:
        month = ((month + 5) % 12) + 1  # shift calendar by 6 months
    if   month in [3, 4, 5]:    return 'Long_Rains'
    elif month in [6, 7, 8]:    return 'Dry_Season'
    elif month in [9, 10, 11]:  return 'Short_Rains'
    else:                       return 'Cool_Dry'

df['SEASON'] = df.apply(lambda r: get_season(r['VISIT_MONTH'], r['COUNTRY']), axis=1)

# --- Real per-patient feature: comorbidity count from actual Synthea conditions ---
# Synthea doesn't model African infectious diseases (malaria/cholera/etc. don't exist in its
# condition modules), so 'conditions' genuinely can't be used to derive the DISEASE label --
# that part has to stay rule-based, there's no way around it with this data source. But
# 'conditions' DOES contain real per-patient chronic-condition history, which is useful as a
# vulnerability/severity feature, so we use it for that instead of leaving 4M+ rows unused.
# NOTE: assumes the patient-id column in conditions.csv is called 'PATIENT' (same convention
# as encounters.csv) -- check this against your actual file and adjust if it differs.
comorbidity_count = conditions.groupby('PATIENT').size().rename('COMORBIDITY_COUNT')
df = df.merge(comorbidity_count, left_on='PATIENT', right_index=True, how='left')
df['COMORBIDITY_COUNT'] = df['COMORBIDITY_COUNT'].fillna(0).astype(int)

# Patient age at time of visit (real, derived from BIRTHDATE) -- age is a genuine risk factor
# for most of these diseases (e.g. under-5s and elderly skew higher-risk).
df['BIRTHDATE'] = pd.to_datetime(df['BIRTHDATE'], errors='coerce')
df['AGE_AT_VISIT'] = ((df['VISIT_DATE'] - df['BIRTHDATE']).dt.days / 365.25).clip(lower=0)

# Assign disease based on patient country (still rule-based -- see note above)
print('Assigning diseases (rule-based by country)...')
df['DISEASE'] = df['COUNTRY'].apply(assign_disease)

print(f'Merged dataset shape: {df.shape}')
print('Disease distribution:')
print(df['DISEASE'].value_counts().head(10))


Merging data...
Assigning diseases (rule-based by country)...
Merged dataset shape: (6744416, 23)
Disease distribution:
DISEASE
Malaria          1170153
Typhoid fever     913183
Tuberculosis      688290
Cholera           640334
Pneumonia         521549
COVID-19          480165
Meningitis        412197
Influenza         394929
Measles           391531
Hepatitis A       347028
Name: count, dtype: int64


## FIX 1: Date Filter (>=2000)

In [9]:
# ===========================================================
# FIX 1: Filter data to 2000+ only
# Reason: Pre-2000 records are irrelevant for a modern early
# warning system and introduce historical noise that hurts the model.
# ===========================================================
print(f"Records before date filter: {len(df):,}")
print(f"Year range before: {df['VISIT_YEAR'].min()} to {df['VISIT_YEAR'].max()}")

df = df[df['VISIT_YEAR'] >= 2000].copy()

print(f"Records after date filter (>=2000): {len(df):,}")
print(f"Year range after: {df['VISIT_YEAR'].min()} to {df['VISIT_YEAR'].max()}")


Records before date filter: 6,744,416
Year range before: 1915 to 2026
Records after date filter (>=2000): 5,881,057
Year range after: 2000 to 2026


## FIX 2: Remove COVID-19 Pre-2019

In [10]:
# ===========================================================
# FIX 2: Remove COVID-19 before 2019
# Reason: COVID-19 was first identified in late 2019.
# Any record labeled COVID-19 before 2019 is an epidemiological
# impossibility that will severely corrupt time-series models.
# Ref: EpiSentinel Data Review Report — Priority Fix #2
# ===========================================================
covid_pre2019_mask = (df['DISEASE'] == 'COVID-19') & (df['VISIT_YEAR'] < 2019)
n_removed = covid_pre2019_mask.sum()
print(f"COVID-19 records before 2019 (removed): {n_removed:,}")

# Reassign those rows to a contextually appropriate disease
# based on their country distribution (excluding COVID-19)
def assign_disease_no_covid(country):
    dist = country_disease_distribution.get(country, country_disease_distribution['DEFAULT']).copy()
    dist.pop('COVID-19', None)
    diseases = list(dist.keys())
    probs = np.array(list(dist.values()))
    probs /= probs.sum()
    return np.random.choice(diseases, p=probs)

df.loc[covid_pre2019_mask, 'DISEASE'] = df.loc[covid_pre2019_mask, 'COUNTRY'].apply(assign_disease_no_covid)
print(f"Reassigned {n_removed:,} pre-2019 COVID-19 records to contextually appropriate diseases.")
print("Disease distribution after fix:")
print(df['DISEASE'].value_counts().head(10))


COVID-19 records before 2019 (removed): 162,712
Reassigned 162,712 pre-2019 COVID-19 records to contextually appropriate diseases.
Disease distribution after fix:
DISEASE
Malaria              1042824
Typhoid fever         824093
Tuberculosis          618566
Cholera               572136
Pneumonia             468996
Meningitis            369457
Influenza             361399
Measles               349125
Hepatitis A           316682
Diarrheal disease     269329
Name: count, dtype: int64


## FIX 3: Fix Geographic Disease Bias

In [11]:
# ===========================================================
# FIX 3: Fix geographic disease distribution bias
# Reason (from review report):
# - Hepatitis A was only in Egypt (0 elsewhere) — artificial
# - HIV/AIDS was only in South Africa (0 elsewhere) — artificial
# - Influenza was only in Egypt + South Africa — artificial
# These biases make the model learn disease = country instead
# of disease = symptom pattern, which is a critical shortcut.
#
# Fix: Reassign a portion of records in affected countries
# so all three diseases appear everywhere at realistic rates.
# Source: WHO AFRO + Africa CDC 2024 disease burden data
# ===========================================================

# Diseases that should appear in ALL countries at some baseline rate
UNIVERSAL_DISEASE_RATES = {
    # disease: {country: probability_to_reassign}
    # We inject these by randomly flipping a small % of records
    'Hepatitis A': {
        # Present across Africa — higher in areas with poor sanitation
        'Nigeria': 0.04, 'Ethiopia': 0.04, 'DRC': 0.04, 'Kenya': 0.04,
        'Tanzania': 0.04, 'Uganda': 0.04, 'Ghana': 0.03, 'Senegal': 0.03,
        'Sudan': 0.03, 'Morocco': 0.04, 'Algeria': 0.04, 'Tunisia': 0.04,
        'South Africa': 0.03, 'Cameroon': 0.03, 'Mozambique': 0.03,
        'Zambia': 0.03, 'Mali': 0.03, 'Zimbabwe': 0.03,
    },
    'HIV/AIDS': {
        # Sub-Saharan Africa has highest global burden — Source: UNAIDS 2023
        'Nigeria': 0.02, 'Ethiopia': 0.02, 'DRC': 0.02, 'Kenya': 0.03,
        'Tanzania': 0.03, 'Uganda': 0.03, 'Ghana': 0.01, 'Senegal': 0.01,
        'Sudan': 0.01, 'Cameroon': 0.02, 'Mozambique': 0.04,
        'Zambia': 0.04, 'Zimbabwe': 0.04,
    },
    'Influenza': {
        # Influenza circulates everywhere year-round in Africa
        'Nigeria': 0.04, 'Ethiopia': 0.04, 'DRC': 0.04, 'Kenya': 0.04,
        'Tanzania': 0.04, 'Uganda': 0.04, 'Ghana': 0.04, 'Senegal': 0.04,
        'Sudan': 0.04, 'Morocco': 0.05, 'Algeria': 0.05, 'Tunisia': 0.05,
        'Cameroon': 0.04, 'Mozambique': 0.04, 'Zambia': 0.04,
        'Mali': 0.04, 'Zimbabwe': 0.04,
    },
}

total_reassigned = 0
for disease, country_rates in UNIVERSAL_DISEASE_RATES.items():
    for country, rate in country_rates.items():
        # Get indices for this country that are NOT already this disease
        mask = (df['COUNTRY'] == country) & (df['DISEASE'] != disease)
        eligible_idx = df[mask].index
        n_to_flip = int(len(eligible_idx) * rate)
        if n_to_flip > 0:
            chosen = np.random.choice(eligible_idx, size=n_to_flip, replace=False)
            df.loc[chosen, 'DISEASE'] = disease
            total_reassigned += n_to_flip

print(f"Total records reassigned for geographic bias fix: {total_reassigned:,}")
print("Updated disease distribution:")
print(df['DISEASE'].value_counts())


Total records reassigned for geographic bias fix: 351,692
Updated disease distribution:
DISEASE
Malaria                959727
Typhoid fever          788478
Tuberculosis           580814
Cholera                529964
Influenza              504172
Hepatitis A            450064
Pneumonia              441496
Meningitis             345961
Measles                318056
Diarrheal disease      257752
COVID-19               246403
Dengue fever           170480
HIV/AIDS               119348
Lassa fever             76605
Ebola virus disease     28956
Mpox                    23072
Leishmaniasis           22535
Rift Valley fever       17174
Name: count, dtype: int64


## FIX 3.1: Hospital / Organization–Location Consistency

In [12]:
# ===========================================================
# FIX 3.1: Hospital-location consistency
# Reason (from audit): ORGANIZATION still holds the raw Synthea org id, which reflects
# the ORIGINAL (US-based) clinic each encounter happened at in the source simulation --
# it has nothing to do with the African CITY/COUNTRY we assigned per patient in Step 6.
# That mismatch is exactly why the audit found 1,202 "hospitals" spanning multiple
# cities/countries: the same US org id gets attached to patients who were randomly
# assigned to many different African cities.
# Fix: derive ORGANIZATION directly from the patient's assigned CITY, the same
# convention already used for injected outbreak records (f'Hospital_{city}'), so every
# organization maps to exactly one city and one country by construction.
# ===========================================================
print(f"ORGANIZATION values before fix: {df['ORGANIZATION'].nunique():,}")
print(f"Hospitals spanning >1 city before fix: "
      f"{(df.groupby('ORGANIZATION')['CITY'].nunique() > 1).sum():,}")

df['ORGANIZATION'] = 'Hospital_' + df['CITY'].astype(str)

print(f"ORGANIZATION values after fix : {df['ORGANIZATION'].nunique():,}")
print(f"Hospitals spanning >1 city after fix : "
      f"{(df.groupby('ORGANIZATION')['CITY'].nunique() > 1).sum():,}")
print(f"Hospitals spanning >1 country after fix: "
      f"{(df.groupby('ORGANIZATION')['COUNTRY'].nunique() > 1).sum():,}")


ORGANIZATION values before fix: 1,220
Hospitals spanning >1 city before fix: 1,202
ORGANIZATION values after fix : 64
Hospitals spanning >1 city after fix : 0
Hospitals spanning >1 country after fix: 0


## FIX 3.2: Age-Appropriate Disease Assignment (Measles Pediatric Skew)

In [13]:
# ===========================================================
# FIX 3.2: Age-appropriate disease assignment
# Reason (from audit): DISEASE is assigned purely from COUNTRY, completely independent
# of AGE_AT_VISIT. That's fine for most of the disease list, but Measles is a textbook
# pediatric disease -- WHO fact sheets put the large majority of cases under 15 -- and
# the audit confirmed the problem: mean age for Measles came out at 47.7, identical to
# every other disease, because age was never a factor in the assignment.
# Fix: 1) move existing Measles cases that are NOT in the target pediatric age range out
#         to another country-appropriate disease (reusing the COVID-safe assigner so we
#         don't undo FIX 2's pre-2019 COVID constraint), then
#      2) pull in enough age-eligible (<15), non-Measles patients to restore roughly the
#         same total Measles case count, so we fix the age skew without collapsing the
#         disease's overall prevalence.
# Source: WHO Measles fact sheet (most cases occur in children under 15)
# ===========================================================

def assign_disease_safe(country, year):
    """Country-based disease draw that respects the FIX 2 constraint: never assigns
    COVID-19 to a pre-2019 record."""
    if year < 2019:
        return assign_disease_no_covid(country)
    return assign_disease(country)

AGE_SENSITIVE_DISEASES = {
    # disease: (min_age, max_age, ) -- pediatric window Measles should concentrate in
    'Measles': (0, 15),
}

for disease, (age_min, age_max) in AGE_SENSITIVE_DISEASES.items():
    disease_mask = df['DISEASE'] == disease
    in_range_mask = df['AGE_AT_VISIT'].between(age_min, age_max)
    n_before = disease_mask.sum()
    mean_age_before = df.loc[disease_mask, 'AGE_AT_VISIT'].mean()

    # Step 1: move out-of-range existing cases to another country-appropriate disease
    wrong_age_idx = df[disease_mask & ~in_range_mask].index
    if len(wrong_age_idx) > 0:
        df.loc[wrong_age_idx, 'DISEASE'] = df.loc[wrong_age_idx].apply(
            lambda r: assign_disease_safe(r['COUNTRY'], r['VISIT_YEAR']), axis=1)

    # Step 2: pull in enough in-range patients to restore the original case count
    n_after_removal = (df['DISEASE'] == disease).sum()
    n_to_add = max(n_before - n_after_removal, 0)
    eligible_idx = df[in_range_mask & (df['DISEASE'] != disease)].index
    if n_to_add > 0 and len(eligible_idx) > 0:
        n_to_add = min(n_to_add, len(eligible_idx))
        chosen = np.random.choice(eligible_idx, size=n_to_add, replace=False)
        df.loc[chosen, 'DISEASE'] = disease

    mean_age_after = df.loc[df['DISEASE'] == disease, 'AGE_AT_VISIT'].mean()
    print(f"{disease}: cases {n_before:,} -> {(df['DISEASE']==disease).sum():,} | "
          f"mean age {mean_age_before:.1f} -> {mean_age_after:.1f}")


Measles: cases 318,056 -> 318,056 | mean age 47.8 -> 11.2


## FIX 3.3: Seasonal Disease Modulation

In [14]:
# ===========================================================
# FIX 3.3: Seasonal disease modulation
# Reason (from audit): DISEASE never depends on SEASON/VISIT_MONTH, so rain-driven
# diseases (Malaria, Cholera) and dry-season diseases (Influenza, Meningitis) showed
# almost no seasonal signal (ratios ~1.0-1.06, all four seasonality checks failed) even
# though this is one of the best-documented, most predictive features for an outbreak
# early-warning model.
# Fix: for each seasonal disease, boost a fraction of non-matching records towards it
# during its peak season(s), and demote a fraction of its existing records away from it
# outside those seasons -- a two-directional nudge, not a full re-draw, so the existing
# rule-based country distribution (and FIXES 2/3/3.2) are preserved everywhere else.
# Source: WHO AFRO seasonal outbreak reporting (rainfall-linked malaria/cholera peaks,
# dry-season meningitis belt, cool-season influenza circulation)
# ===========================================================

SEASONAL_DISEASE_BOOST = {
    'Malaria':    {'peak_seasons': ['Long_Rains', 'Short_Rains'], 'boost_rate': 0.30, 'demote_rate': 0.15},
    'Cholera':    {'peak_seasons': ['Long_Rains', 'Short_Rains'], 'boost_rate': 0.28, 'demote_rate': 0.14},
    'Influenza':  {'peak_seasons': ['Cool_Dry'],                  'boost_rate': 0.35, 'demote_rate': 0.18},
    'Meningitis': {'peak_seasons': ['Dry_Season'],                'boost_rate': 0.35, 'demote_rate': 0.18},
}

total_seasonal_reassigned = 0
for disease, cfg in SEASONAL_DISEASE_BOOST.items():
    peak_seasons = cfg['peak_seasons']

    # Boost: convert some non-disease records IN the peak season(s) to this disease
    boost_idx = df[df['SEASON'].isin(peak_seasons) & (df['DISEASE'] != disease)].index
    n_boost = int(len(boost_idx) * cfg['boost_rate'])
    if n_boost > 0:
        chosen = np.random.choice(boost_idx, size=n_boost, replace=False)
        df.loc[chosen, 'DISEASE'] = disease
        total_seasonal_reassigned += n_boost

    # Demote: convert some disease records OUTSIDE the peak season(s) to something else
    # (COVID-safe, so we never violate FIX 2's pre-2019 constraint)
    demote_idx = df[(~df['SEASON'].isin(peak_seasons)) & (df['DISEASE'] == disease)].index
    n_demote = int(len(demote_idx) * cfg['demote_rate'])
    if n_demote > 0:
        chosen = np.random.choice(demote_idx, size=n_demote, replace=False)
        df.loc[chosen, 'DISEASE'] = df.loc[chosen].apply(
            lambda r: assign_disease_safe(r['COUNTRY'], r['VISIT_YEAR']), axis=1)
        total_seasonal_reassigned += n_demote

print(f"Total records reassigned for seasonal signal: {total_seasonal_reassigned:,}")

print("\nQuick check -- cases per month for the 4 seasonal diseases:")
for disease in SEASONAL_DISEASE_BOOST:
    print(f"  {disease}:")
    print(df[df['DISEASE'] == disease].groupby('VISIT_MONTH').size().to_string())


Total records reassigned for seasonal signal: 2,650,238

Quick check -- cases per month for the 4 seasonal diseases:
  Malaria:
VISIT_MONTH
1      48593
2      47488
3     162366
4     161201
5     154482
6      47713
7      47100
8      47247
9     140339
10    144391
11    141151
12     47610
  Cholera:
VISIT_MONTH
1      26897
2      26090
3     175328
4     173380
5     168608
6      26525
7      26262
8      26236
9     151820
10    157108
11    152299
12     26477
  Influenza:
VISIT_MONTH
1     180743
2     175884
3      18771
4      18618
5      17792
6      40014
7      38902
8      39036
9      16298
10     16760
11     16541
12    176561
  Meningitis:
VISIT_MONTH
1      33815
2      33579
3      13580
4      13378
5      12960
6     172045
7     170047
8     170218
9      11817
10     12031
11     11846
12     33139


In [15]:
# ===========================================================
# FIX 4: COMORBIDITY_COUNT — Realistic Rescaling
# Problem: Raw Synthea values are 0-549 (counts ALL condition codes
# ever assigned including admin codes, not just chronic comorbidities).
# Clip(0,10) made 94% of records = 10 — not realistic.
# Fix: Use percentile-based rescaling to 0-10 range so the
# distribution is spread realistically, not all piled at 10.
# Clinical reference: most patients have 0-3 comorbidities;
# very complex patients have up to 8-10. Source: Elixhauser (1998).
# ===========================================================
print(f"COMORBIDITY_COUNT before fix — max: {df['COMORBIDITY_COUNT'].max()}, mean: {df['COMORBIDITY_COUNT'].mean():.1f}")
print(f"Distribution before fix:\n{df['COMORBIDITY_COUNT'].value_counts().sort_index().head(15)}")

# Percentile-based rescale: map 0-95th percentile → 0-10
p95 = df['COMORBIDITY_COUNT'].quantile(0.95)
df['COMORBIDITY_COUNT'] = (
    df['COMORBIDITY_COUNT']
    .clip(upper=p95)
    .pipe(lambda x: (x / p95 * 10).round().astype(int))
    .clip(0, 10)
)

print(f"\nCOMORBIDITY_COUNT after fix — max: {df['COMORBIDITY_COUNT'].max()}, mean: {df['COMORBIDITY_COUNT'].mean():.1f}")
print(f"Distribution after fix:\n{df['COMORBIDITY_COUNT'].value_counts().sort_index()}")


COMORBIDITY_COUNT before fix — max: 549, mean: 73.9
Distribution before fix:
COMORBIDITY_COUNT
1       663
2      1720
3      2977
4      4013
5      5823
6      8271
7     10417
8     13760
9     19756
10    23417
11    30613
12    38468
13    45945
14    54741
15    62183
Name: count, dtype: int64

COMORBIDITY_COUNT after fix — max: 10, mean: 2.8
Distribution after fix:
COMORBIDITY_COUNT
0      159898
1     2425437
2     1392148
3      405369
4      360177
5      282145
6      172754
7      128508
8      120685
9       92824
10     341112
Name: count, dtype: int64


## FIX 5: Fix AGE_AT_VISIT

In [16]:
# ===========================================================
# FIX 5: Fix impossible AGE_AT_VISIT values
# Reason: Newborns (age=0 days) diagnosed with COVID-19 on
# day of birth, and patients > 110 years old are data artifacts
# from Synthea's simulation engine, not real cases.
# Min age for infectious disease visit = 0.08 years (~1 month)
# Max realistic age = 100 years
# ===========================================================
print(f"AGE_AT_VISIT before fix — min: {df['AGE_AT_VISIT'].min():.2f}, max: {df['AGE_AT_VISIT'].max():.2f}")
print(f"Records with age < 0.08 (under 1 month): {(df['AGE_AT_VISIT'] < 0.08).sum():,}")
print(f"Records with age > 100: {(df['AGE_AT_VISIT'] > 100).sum():,}")

df['AGE_AT_VISIT'] = df['AGE_AT_VISIT'].clip(0.08, 100)

# Fill missing AGE_AT_VISIT (294,220 records from outbreak injection) 
# with country-level median age — better than leaving NaN
country_median_age = df.groupby('COUNTRY')['AGE_AT_VISIT'].median()
df['AGE_AT_VISIT'] = df.apply(
    lambda row: country_median_age.get(row['COUNTRY'], 30.0) 
    if pd.isna(row['AGE_AT_VISIT']) else row['AGE_AT_VISIT'],
    axis=1
)

print(f"AGE_AT_VISIT after fix  — min: {df['AGE_AT_VISIT'].min():.2f}, max: {df['AGE_AT_VISIT'].max():.2f}")
print(f"Missing AGE_AT_VISIT remaining: {df['AGE_AT_VISIT'].isna().sum()}")


AGE_AT_VISIT before fix — min: 0.00, max: 110.89
Records with age < 0.08 (under 1 month): 11,619
Records with age > 100: 58,403
AGE_AT_VISIT after fix  — min: 0.08, max: 100.00
Missing AGE_AT_VISIT remaining: 0


## FIX 6: Add VISIT_NUMBER

In [17]:
# ===========================================================
# FIX 6: Add VISIT_NUMBER per patient
# Reason: 6.6M+ rows share the same PATIENT ID (repeat visits).
# Without a visit number, models treating each row as independent
# risk data leakage — the model may memorize patient-level
# patterns instead of learning population-level outbreak signals.
# ===========================================================
df = df.sort_values(['PATIENT', 'VISIT_DATE'])
df['VISIT_NUMBER'] = df.groupby('PATIENT').cumcount() + 1
print(f"VISIT_NUMBER added. Distribution of visit counts:")
print(df.groupby('PATIENT')['VISIT_NUMBER'].max().describe())


VISIT_NUMBER added. Distribution of visit counts:
count    110383.000000
mean         53.278648
std          81.210982
min           1.000000
25%          23.000000
50%          33.000000
75%          53.000000
max         948.000000
Name: VISIT_NUMBER, dtype: float64


## Step 9: Add Binary Symptom Columns
Each symptom becomes its own column (1 = present, 0 = absent)  
This format is optimal for ML models — enables syndrome detection and feature engineering

In [18]:
print('Generating binary symptom columns...')
print('This may take a few minutes for large datasets...')

# Generate symptom vectors for each row
symptom_records = [get_symptom_vector(disease) for disease in df['DISEASE']]
symptoms_df = pd.DataFrame(symptom_records, index=df.index)

# Join symptom columns to main dataframe
df = pd.concat([df, symptoms_df], axis=1)

print(f'Done. Symptom columns added: {len(ALL_SYMPTOMS)}')
print('Sample symptom prevalence:')
print(df[['sym_fever', 'sym_cough', 'sym_diarrhea', 'sym_rash', 'sym_hemorrhage']].mean().round(3))

Generating binary symptom columns...
This may take a few minutes for large datasets...
Done. Symptom columns added: 28
Sample symptom prevalence:
sym_fever         0.741
sym_cough         0.308
sym_diarrhea      0.260
sym_rash          0.121
sym_hemorrhage    0.052
dtype: float64


## FIX 7: Regenerate Symptoms After Disease Fixes

In [19]:
# ===========================================================
# FIX 7: Regenerate symptom columns after disease fixes
# Reason: Fixes 2, 3, 3.2, and 3.3 changed DISEASE for many records but
# the symptom columns still reflect the old (wrong) disease.
# We must regenerate symptoms to match the corrected diseases.
# ===========================================================
print("Regenerating symptom columns after disease fixes...")
print("(This may take a few minutes for large datasets)")

sym_cols = [c for c in df.columns if c.startswith("sym_")]
df = df.drop(columns=sym_cols)

symptom_records = [get_symptom_vector(disease) for disease in df['DISEASE']]
symptoms_df = pd.DataFrame(symptom_records, index=df.index)
df = pd.concat([df, symptoms_df], axis=1)

for col in ALL_SYMPTOMS:
    df[col] = df[col].fillna(0).astype(int)

print(f"Symptoms regenerated for {len(df):,} records.")
print("Sample symptom prevalence:")
print(df[[s for s in ALL_SYMPTOMS[:5]]].mean().round(3))


Regenerating symptom columns after disease fixes...
(This may take a few minutes for large datasets)
Symptoms regenerated for 5,881,057 records.
Sample symptom prevalence:
sym_fever       0.741
sym_chills      0.222
sym_headache    0.496
sym_fatigue     0.476
sym_vomiting    0.416
dtype: float64


## Step 10: Simulate Outbreak Events
Inject realistic outbreak patterns based on historical African outbreaks  
**Purpose:** Teaches the model what a real outbreak looks like — sudden spike in cases in a specific location and time  
**Source:** WHO outbreak reports + Africa CDC historical data

### Step 10 (updated): Realistic, Varied Outbreak & Noise Simulation
Instead of 12 fixed, hand-picked outbreaks, this now generates:
- **12 historically-anchored scenarios** (kept from the original script, for grounding)
- **150 randomly-generated outbreaks** with varied location, disease, timing, magnitude, and
  curve shape, so the model sees hundreds of distinct examples instead of memorizing a
  handful of exact patterns
- **80 background noise bumps** -- short, small, non-outbreak spikes so the model has to learn
  to actually distinguish a real outbreak from ordinary fluctuation, not just flag any rise
- **Reporting delay** on injected records, longer in smaller/lower-resource cities (a real,
  well-documented surveillance-data effect)


In [20]:
# IS_OUTBREAK flag for normal records
df['IS_OUTBREAK'] = 0
df['IS_BACKGROUND_NOISE'] = 0

RNG_SEED = 42
rng = np.random.RandomState(RNG_SEED)

# Diseases capable of producing discrete outbreak spikes. Chronic diseases (HIV/AIDS,
# Tuberculosis, Hepatitis A) build up slowly in a population and don't show sudden
# "outbreak curves" the way these do -- excluding them keeps the outbreak label meaningful.
EPIDEMIC_PRONE_DISEASES = ['Cholera', 'Ebola virus disease', 'Lassa fever', 'Meningitis',
                           'Measles', 'Mpox', 'Dengue fever', 'COVID-19', 'Yellow fever',
                           'Rift Valley fever', 'Influenza', 'Malaria']

# A handful of real, historically-grounded events (kept from the original script) --
# these anchor the simulation to actual documented outbreaks. Everything else is
# randomly generated to give the model many more varied examples instead of memorizing
# just these 12 exact shapes/locations/dates.
anchored_scenarios = [
    {'city': 'Kinshasa',     'country': 'DRC',      'disease': 'Ebola virus disease', 'start': '2020-08-01', 'duration': 90,  'extra_cases': 3000},
    {'city': 'Lagos',        'country': 'Nigeria',  'disease': 'Lassa fever',         'start': '2021-01-01', 'duration': 89,  'extra_cases': 2500},
    {'city': 'Nairobi',      'country': 'Kenya',    'disease': 'Cholera',             'start': '2021-04-01', 'duration': 90,  'extra_cases': 4000},
    {'city': 'Addis Ababa',  'country': 'Ethiopia', 'disease': 'Malaria',             'start': '2021-09-01', 'duration': 90,  'extra_cases': 5000},
    {'city': 'Kampala',      'country': 'Uganda',   'disease': 'Ebola virus disease', 'start': '2022-09-01', 'duration': 90,  'extra_cases': 2000},
    {'city': 'Khartoum',     'country': 'Sudan',    'disease': 'Cholera',             'start': '2022-07-01', 'duration': 91,  'extra_cases': 3500},
    {'city': 'Cairo',        'country': 'Egypt',    'disease': 'COVID-19',            'start': '2021-01-01', 'duration': 119, 'extra_cases': 6000},
    {'city': 'Goma',         'country': 'DRC',      'disease': 'Mpox',                'start': '2023-01-01', 'duration': 119, 'extra_cases': 2500},
    {'city': 'Dar es Salaam','country': 'Tanzania', 'disease': 'Dengue fever',        'start': '2022-03-01', 'duration': 121, 'extra_cases': 3000},
    {'city': 'Kano',         'country': 'Nigeria',  'disease': 'Meningitis',          'start': '2021-12-01', 'duration': 89,  'extra_cases': 2000},
    {'city': 'Mansoura',     'country': 'Egypt',    'disease': 'Typhoid fever',       'start': '2022-05-01', 'duration': 91,  'extra_cases': 1500},
    {'city': 'Lusaka',       'country': 'Zambia',   'disease': 'Cholera',             'start': '2023-01-01', 'duration': 89,  'extra_cases': 3000},
]
for s in anchored_scenarios:
    s['start'] = pd.Timestamp(s['start'])
    s['peak_frac'] = 0.35
    s['shape'] = 3.0
    s['spread_prob'] = 1.0

DATA_START = df['VISIT_DATE'].min()
DATA_END   = df['VISIT_DATE'].max()

def generate_random_outbreaks(n_outbreaks, rng):
    """Randomized outbreak generator -- varies location, disease, timing, magnitude, and
    curve shape every time, so the model sees hundreds of distinct outbreak shapes instead
    of memorizing a small fixed set."""
    total_days = (DATA_END - DATA_START).days
    scenarios = []
    for _ in range(n_outbreaks):
        city_info = african_cities[rng.choice(len(african_cities), p=city_weights)]
        country = city_info['COUNTRY']
        options = [d for d in EPIDEMIC_PRONE_DISEASES
                   if d in country_disease_distribution.get(country, country_disease_distribution['DEFAULT'])]
        if not options:
            options = EPIDEMIC_PRONE_DISEASES
        duration = int(rng.uniform(20, 150))
        start_offset = int(rng.uniform(0, max(total_days - duration, 1)))
        # Magnitude scales with city population weight (bigger cities -> bigger outbreaks on
        # average) but with log-normal randomness so it isn't perfectly predictable from
        # weight alone -- real outbreak sizes are noisy even in similarly-sized cities.
        magnitude_base = 300 + city_info['WEIGHT'] * 250
        scenarios.append({
            'city': city_info['CITY'], 'country': country,
            'disease': rng.choice(options),
            'start': DATA_START + pd.Timedelta(days=start_offset),
            'duration': duration,
            'extra_cases': int(rng.lognormal(mean=np.log(magnitude_base), sigma=0.55)),
            'peak_frac': rng.uniform(0.2, 0.5),
            'shape': rng.uniform(2.0, 5.0),
            'spread_prob': rng.uniform(0, 1),
        })
    return scenarios

def generate_background_bumps(n_bumps, rng):
    """Short, small, non-outbreak spikes in ordinary disease incidence. Without these, a
    classifier can take the shortcut of flagging ANY rise as an outbreak. These bumps are
    intentionally shorter (7-21 days vs 20-150) and smaller than real outbreaks so the model
    has to learn to actually tell the difference instead of using magnitude/duration as a
    trivial giveaway."""
    total_days = (DATA_END - DATA_START).days
    bumps = []
    for _ in range(n_bumps):
        city_info = african_cities[rng.choice(len(african_cities), p=city_weights)]
        country = city_info['COUNTRY']
        dist = country_disease_distribution.get(country, country_disease_distribution['DEFAULT'])
        duration = int(rng.uniform(7, 21))
        start_offset = int(rng.uniform(0, max(total_days - duration, 1)))
        bumps.append({
            'city': city_info['CITY'], 'country': country,
            'disease': rng.choice(list(dist.keys())),
            'start': DATA_START + pd.Timedelta(days=start_offset),
            'duration': duration,
            'extra_cases': int(rng.uniform(30, 150)),
        })
    return bumps

outbreak_scenarios = anchored_scenarios + generate_random_outbreaks(150, rng)
background_bumps   = generate_background_bumps(80, rng)

def epidemic_daily_weights(n_days, peak_frac=0.35, shape=3.0):
    """Gamma-shaped daily case-count curve: slow rise -> peak -> longer decline tail --
    not a flat random spread across the window."""
    peak_day = max(n_days * peak_frac, 1)
    scale = peak_day / (shape - 1) if shape > 1 else peak_day
    days = np.arange(1, n_days + 1, dtype=float)
    weights = (days ** (shape - 1)) * np.exp(-days / scale)
    return weights / weights.sum()

def add_reporting_delay(visit_dates, city_weight, rng):
    """Real health systems don't report cases same-day. Lower city WEIGHT (smaller, more
    rural/lower-resource cities) gets a longer average delay -- a real, well-documented
    surveillance-data pattern."""
    scale = 2 + (10 - city_weight) * 0.6
    delay_days = rng.exponential(scale=scale, size=len(visit_dates)).round().astype(int)
    return pd.to_datetime(visit_dates) + pd.to_timedelta(delay_days, unit='D')

city_lookup = {c['CITY']: c for c in african_cities}
cities_by_country = {}
for c in african_cities:
    cities_by_country.setdefault(c['COUNTRY'], []).append(c['CITY'])

def make_case_block(city, country, disease, visit_dates, is_outbreak, is_background_noise=0, is_seed=False):
    """Vectorized generation of one block of injected records (outbreak, spread, or
    background-noise bump)."""
    n = len(visit_dates)
    city_info = city_lookup[city]
    syms_df = pd.DataFrame([get_symptom_vector(disease) for _ in range(n)])
    reported_dates = add_reporting_delay(visit_dates, city_info['WEIGHT'], rng)
    block = pd.DataFrame({
        'PATIENT':             [f'OB_{x}' for x in rng.randint(1_000_000, 9_999_999, size=n)],
        'ORGANIZATION':        f'Hospital_{city}',
        'VISIT_DATE':          visit_dates,
        'REPORTED_DATE':       reported_dates,
        'ENCOUNTERCLASS':      'emergency' if is_outbreak else 'ambulatory',
        'GENDER':              rng.choice(['M', 'F'], size=n),
        'CITY':                city,
        'STATE':               city_info['STATE'],
        'COUNTRY':             country,
        'LAT':                 city_info['LAT'],
        'LON':                 city_info['LON'],
        'DISEASE':             disease,
        'IS_OUTBREAK':         is_outbreak,
        'IS_SECONDARY_SPREAD': int(is_seed),
        'IS_BACKGROUND_NOISE': is_background_noise,
    })
    block['VISIT_YEAR']  = pd.DatetimeIndex(block['VISIT_DATE']).year
    block['VISIT_MONTH'] = pd.DatetimeIndex(block['VISIT_DATE']).month
    block['VISIT_WEEK']  = pd.DatetimeIndex(block['VISIT_DATE']).isocalendar().week.values
    block['VISIT_DAY']   = pd.DatetimeIndex(block['VISIT_DATE']).day_of_year
    block['SEASON']      = block['VISIT_MONTH'].apply(lambda m: get_season(m, country))
    return pd.concat([block.reset_index(drop=True), syms_df.reset_index(drop=True)], axis=1)

outbreak_blocks = []

for scenario in outbreak_scenarios:
    duration = scenario['duration']
    weights = epidemic_daily_weights(duration, scenario['peak_frac'], scenario['shape'])
    day_offsets = rng.choice(duration, size=scenario['extra_cases'], p=weights)
    visit_dates = scenario['start'] + pd.to_timedelta(day_offsets, unit='D')

    outbreak_blocks.append(make_case_block(
        scenario['city'], scenario['country'], scenario['disease'], visit_dates, is_outbreak=1
    ))

    # Spatial spread -- not guaranteed every time (spread_prob), matching how not every real
    # outbreak crosses into neighboring areas.
    if rng.uniform(0, 1) < scenario['spread_prob']:
        same_country_cities = [c for c in cities_by_country.get(scenario['country'], []) if c != scenario['city']]
        if same_country_cities:
            n_targets = min(rng.randint(1, 3), len(same_country_cities))
            targets = rng.choice(same_country_cities, size=n_targets, replace=False)
            spread_start = scenario['start'] + pd.Timedelta(days=int(duration * 0.35))
            spread_duration = max(int(duration * 0.5), 7)
            spread_weights = epidemic_daily_weights(spread_duration, peak_frac=0.4)
            for target_city in targets:
                spread_cases = max(int(scenario['extra_cases'] * rng.uniform(0.08, 0.20)), 15)
                offsets = rng.choice(spread_duration, size=spread_cases, p=spread_weights)
                spread_dates = spread_start + pd.to_timedelta(offsets, unit='D')
                outbreak_blocks.append(make_case_block(
                    target_city, scenario['country'], scenario['disease'], spread_dates,
                    is_outbreak=1, is_seed=True
                ))

for bump in background_bumps:
    weights = epidemic_daily_weights(bump['duration'], peak_frac=rng.uniform(0.3, 0.6), shape=rng.uniform(1.5, 3.0))
    day_offsets = rng.choice(bump['duration'], size=bump['extra_cases'], p=weights)
    visit_dates = bump['start'] + pd.to_timedelta(day_offsets, unit='D')
    outbreak_blocks.append(make_case_block(
        bump['city'], bump['country'], bump['disease'], visit_dates,
        is_outbreak=0, is_background_noise=1
    ))

outbreak_df = pd.concat(outbreak_blocks, ignore_index=True)
print(f'Injected records total: {len(outbreak_df):,}')
print(f'  True outbreak cases      : {int((outbreak_df["IS_OUTBREAK"]==1).sum()):,} '
      f'(scenarios: {len(outbreak_scenarios)}, of which {len(anchored_scenarios)} historically-anchored)')
print(f'  Secondary spread cases   : {int(outbreak_df["IS_SECONDARY_SPREAD"].sum()):,}')
print(f'  Background noise bumps   : {int((outbreak_df["IS_BACKGROUND_NOISE"]==1).sum()):,} '
      f'(scenarios: {len(background_bumps)})')
print(outbreak_df['DISEASE'].value_counts().head(10))


Injected records total: 294,220
  True outbreak cases      : 286,584 (scenarios: 162, of which 12 historically-anchored)
  Secondary spread cases   : 31,267
  Background noise bumps   : 7,636 (scenarios: 80)
DISEASE
COVID-19               59089
Cholera                58458
Measles                37957
Malaria                35370
Influenza              31546
Meningitis             27887
Dengue fever           16441
Lassa fever            11380
Ebola virus disease     6343
Mpox                    3376
Name: count, dtype: int64


## Step 11: Combine & Save Master Dataset

In [21]:
print('Combining all data...')

# Align columns between main df and outbreak df (union of both, order preserved)
all_cols = list(dict.fromkeys(list(df.columns) + list(outbreak_df.columns)))
for col in all_cols:
    if col not in df.columns:
        df[col] = np.nan
    if col not in outbreak_df.columns:
        outbreak_df[col] = np.nan

master = pd.concat([df[all_cols], outbreak_df[all_cols]], ignore_index=True)

# Sort by visit date -- this now works because VISIT_DATE is consistently tz-naive everywhere
master = master.sort_values('VISIT_DATE').reset_index(drop=True)

# Fill NaNs introduced by the concat/align step
for col in ALL_SYMPTOMS:
    master[col] = master[col].fillna(0).astype(int)
master['IS_OUTBREAK'] = master['IS_OUTBREAK'].fillna(0).astype(int)
master['IS_SECONDARY_SPREAD'] = master['IS_SECONDARY_SPREAD'].fillna(0).astype(int)
master['IS_BACKGROUND_NOISE'] = master['IS_BACKGROUND_NOISE'].fillna(0).astype(int)
master['REPORTED_DATE'] = master['REPORTED_DATE'].fillna(master['VISIT_DATE'])
master['COMORBIDITY_COUNT'] = master['COMORBIDITY_COUNT'].fillna(0).astype(int)


Combining all data...


## Post-Merge Fixes — Applied to Master Dataset

In [22]:
# ===========================================================
# POST-MERGE FIXES on master dataset
# ===========================================================

# FIX A: ENCOUNTERCLASS Data Leakage
# Problem: All injected outbreak records have ENCOUNTERCLASS='emergency'
# making it a 57% predictor of IS_OUTBREAK — critical data leakage.
# Fix: Assign realistic mixed encounter classes to outbreak records.
# In real outbreaks, patients arrive via multiple routes.
# ===========================================================
print("Fixing ENCOUNTERCLASS leakage in outbreak records...")

OUTBREAK_ENCOUNTER_DIST = {
    "ambulatory": 0.40,
    "emergency":  0.20,
    "inpatient":  0.15,
    "outpatient": 0.20,
    "urgentcare": 0.05,
}
enc_classes = list(OUTBREAK_ENCOUNTER_DIST.keys())
enc_probs   = list(OUTBREAK_ENCOUNTER_DIST.values())

outbreak_mask = master["IS_OUTBREAK"] == 1
n_outbreak = outbreak_mask.sum()
rng_enc = np.random.RandomState(99)
master.loc[outbreak_mask, "ENCOUNTERCLASS"] = rng_enc.choice(
    enc_classes, size=n_outbreak, p=enc_probs
)
enc_rate = pd.crosstab(master["ENCOUNTERCLASS"], master["IS_OUTBREAK"], normalize="index")
emergency_outbreak_rate = enc_rate.loc["emergency", 1] if "emergency" in enc_rate.index else 0
print(f"✅ Emergency→Outbreak rate: {emergency_outbreak_rate:.1%} (was 57%, target < 30%)")

# FIX C: Ensure no COVID-19 before 2019 in master (catches any slipthrough)
master["VISIT_DATE"] = pd.to_datetime(master["VISIT_DATE"], errors="coerce")
covid_pre2019 = (master["DISEASE"] == "COVID-19") & (master["VISIT_DATE"].dt.year < 2019)
n_slip = covid_pre2019.sum()
if n_slip > 0:
    print(f"\nFixing {n_slip:,} COVID-19 slipthrough records (pre-2019)...")
    master.loc[covid_pre2019, "DISEASE"] = np.random.choice(
        ["Influenza", "Meningitis", "Cholera", "Malaria"], size=n_slip
    )
    # Regenerate symptoms for changed records
    for idx in master.index[covid_pre2019]:
        disease = master.at[idx, "DISEASE"]
        syms = get_symptom_vector(disease)
        for sym, val in syms.items():
            if sym in master.columns:
                master.at[idx, sym] = val
    print(f"✅ COVID-19 pre-2019 fixed. Remaining: {((master['DISEASE']=='COVID-19') & (master['VISIT_DATE'].dt.year < 2019)).sum()}")
else:
    print(f"\n✅ No COVID-19 pre-2019 slipthrough.")

# FIX D: Add HOSPITAL_NAME column locked to one city per hospital
print("\nAdding HOSPITAL_NAME locked to city...")
master["HOSPITAL_NAME"] = "Hospital_" + master["CITY"].astype(str)
print(f"✅ HOSPITAL_NAME added. Unique hospitals: {master['HOSPITAL_NAME'].nunique()}")

print("\n✅ All post-merge fixes applied to master dataset.")

# FIX B: Fill missing AGE_AT_VISIT in outbreak records
# Outbreak records have no BIRTHDATE → AGE is NaN
# Fill with realistic age distribution per disease
# Source: WHO age-stratified disease burden data
print("\nFilling missing AGE_AT_VISIT in outbreak records...")

DISEASE_AGE_PARAMS = {
    'Malaria':              (22, 18),
    'Cholera':              (28, 20),
    'Typhoid fever':        (18, 12),
    'COVID-19':             (52, 22),
    'Measles':              (5,  4),
    'Meningitis':           (15, 12),
    'Dengue fever':         (25, 16),
    'Lassa fever':          (30, 15),
    'Ebola virus disease':  (32, 14),
    'Mpox':                 (30, 12),
    'Influenza':            (40, 25),
    'Rift Valley fever':    (35, 18),
    'Tuberculosis':         (38, 18),
    'HIV/AIDS':             (35, 12),
    'Hepatitis A':          (20, 15),
    'Pneumonia':            (45, 28),
    'Diarrheal disease':    (10, 12),
    'Leishmaniasis':        (25, 18),
}

age_missing = master['AGE_AT_VISIT'].isna()
print(f"Missing AGE_AT_VISIT: {age_missing.sum():,}")

rng_age = np.random.RandomState(77)
for disease, (mu, sigma) in DISEASE_AGE_PARAMS.items():
    mask = age_missing & (master['DISEASE'] == disease)
    n = mask.sum()
    if n > 0:
        ages = rng_age.normal(mu, sigma, n).clip(0.08, 100).round(1)
        master.loc[mask, 'AGE_AT_VISIT'] = ages

# Fill any remaining with overall median
still_missing = master['AGE_AT_VISIT'].isna()
if still_missing.sum() > 0:
    master.loc[still_missing, 'AGE_AT_VISIT'] = master['AGE_AT_VISIT'].median()

print(f"Missing AGE_AT_VISIT after fix: {master['AGE_AT_VISIT'].isna().sum()}")


Fixing ENCOUNTERCLASS leakage in outbreak records...
✅ Emergency→Outbreak rate: 20.9% (was 57%, target < 30%)

Fixing 33,686 COVID-19 slipthrough records (pre-2019)...
✅ COVID-19 pre-2019 fixed. Remaining: 0

Adding HOSPITAL_NAME locked to city...
✅ HOSPITAL_NAME added. Unique hospitals: 64

✅ All post-merge fixes applied to master dataset.

Filling missing AGE_AT_VISIT in outbreak records...
Missing AGE_AT_VISIT: 294,220
Missing AGE_AT_VISIT after fix: 0


## Save Final Master Dataset (After All Post-Merge Fixes)

In [23]:
# ===========================================================
# SAVE FINAL MASTER DATASET
# Must run AFTER all post-merge fixes are applied
# ===========================================================
OUTPUT_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\master_dataset.csv'
print('Saving final master dataset after all fixes...')
master.to_csv(OUTPUT_PATH, index=False)

print(f'\n✅ MASTER DATASET SAVED SUCCESSFULLY!')
print(f'Total records       : {len(master):,}')
print(f'Outbreak records    : {int(master["IS_OUTBREAK"].sum()):,}')
print(f'Normal records      : {int((master["IS_OUTBREAK"]==0).sum()):,}')
print(f'Total columns       : {len(master.columns)}')
print(f'Diseases            : {master["DISEASE"].nunique()}')
print(f'Cities              : {master["CITY"].nunique()}')
print(f'Countries           : {master["COUNTRY"].nunique()}')
print(f'Missing AGE_AT_VISIT: {master["AGE_AT_VISIT"].isna().sum()}')
covid_pre2019 = ((master["DISEASE"] == "COVID-19") & (pd.to_datetime(master["VISIT_DATE"]).dt.year < 2019)).sum()
print(f'COVID-19 pre-2019   : {covid_pre2019}')
print(f'HOSPITAL_NAME exists: {"HOSPITAL_NAME" in master.columns}')
print(f'Date range          : {master["VISIT_DATE"].min()} to {master["VISIT_DATE"].max()}')
print(f'\nFile saved to: {OUTPUT_PATH}')


Saving final master dataset after all fixes...

✅ MASTER DATASET SAVED SUCCESSFULLY!
Total records       : 6,175,277
Outbreak records    : 286,584
Normal records      : 5,888,693
Total columns       : 56
Diseases            : 18
Cities              : 64
Countries           : 19
Missing AGE_AT_VISIT: 0
COVID-19 pre-2019   : 0
HOSPITAL_NAME exists: True
Date range          : 2000-01-01 00:16:23 to 2026-07-10 10:00:13

File saved to: C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\master_dataset.csv


## Step 12: Validation & Summary

In [24]:
print('='*50)
print('MASTER DATASET VALIDATION')
print('='*50)

print('\n1. Shape:', master.shape)

print('\n2. Disease distribution:')
print(master['DISEASE'].value_counts())

print('\n3. Top 10 cities:')
print(master['CITY'].value_counts().head(10))

print('\n4. Season distribution:')
print(master['SEASON'].value_counts())

print('\n5. Outbreak events by disease:')
print(master[master['IS_OUTBREAK']==1]['DISEASE'].value_counts())

print('\n6. Symptom prevalence (top 10):')
sym_cols = [c for c in master.columns if c.startswith('sym_')]
print(master[sym_cols].mean().sort_values(ascending=False).head(10).round(3))

print('\n7. Missing values:')
missing = master.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values!')

print('\n✅ Validation complete!')

MASTER DATASET VALIDATION

1. Shape: (6175277, 56)

2. Disease distribution:
DISEASE
Malaria                1233329
Cholera                1204072
Influenza               795854
Meningitis              724778
Typhoid fever           471096
Tuberculosis            352021
Pneumonia               267718
Hepatitis A             254668
Measles                 229378
COVID-19                170322
Diarrheal disease       153417
Dengue fever            122729
HIV/AIDS                 68725
Lassa fever              59098
Ebola virus disease      24415
Mpox                     17912
Leishmaniasis            14068
Rift Valley fever        11677
Name: count, dtype: int64

3. Top 10 cities:
CITY
Cairo            443001
Lagos            354395
Alexandria       256827
Addis Ababa      212588
Nairobi          205788
Kinshasa         204221
Giza             199851
Kano             181612
Dar es Salaam    167522
Johannesburg     165479
Name: count, dtype: int64

4. Season distribution:
SEASON
Long_Rain

## Final Validation — Post-Fix Quality Checks

In [3]:
# ===========================================================
# Load from saved CSV — validate the ACTUAL file, not memory
# ===========================================================
OUTPUT_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\master_dataset.csv'
print(f"Loading saved CSV for validation...")
master = pd.read_csv(OUTPUT_PATH, parse_dates=['VISIT_DATE'])
print(f"✅ Loaded: {len(master):,} records from disk")
print(f"   Columns: {len(master.columns)}")
print()

df = master.copy()

# ===========================================================
# FINAL VALIDATION — Post-fix quality checks
# Verifies all 7 fixes were applied correctly
# ===========================================================
print("=" * 60)
print("FINAL VALIDATION REPORT")
print("=" * 60)

passed = 0
failed = 0

def check(name, condition, detail=""):
    global passed, failed
    status = "✅ PASS" if condition else "❌ FAIL"
    if condition:
        passed += 1
    else:
        failed += 1
    print(f"{status} | {name}")
    if detail:
        print(f"       {detail}")

print("\n--- FIX 1: Date Filter ---")
check("No records before 2000",
      df['VISIT_YEAR'].min() >= 2000,
      f"Min year: {df['VISIT_YEAR'].min()}")

print("\n--- FIX 2: COVID-19 Timeline ---")
covid_pre2019 = ((df['DISEASE'] == 'COVID-19') & (df['VISIT_YEAR'] < 2019)).sum()
check("No COVID-19 before 2019",
      covid_pre2019 == 0,
      f"COVID-19 pre-2019 records: {covid_pre2019}")

print("\n--- FIX 3: Geographic Disease Distribution ---")
for disease in ['Hepatitis A', 'HIV/AIDS', 'Influenza']:
    countries_with_disease = df[df['DISEASE'] == disease]['COUNTRY'].nunique()
    check(f"{disease} present in multiple countries",
          countries_with_disease >= 5,
          f"Countries with {disease}: {countries_with_disease}")

print("\n--- FIX 4: COMORBIDITY_COUNT ---")
check("COMORBIDITY_COUNT max <= 10",
      df['COMORBIDITY_COUNT'].max() <= 10,
      f"Max value: {df['COMORBIDITY_COUNT'].max()}")

print("\n--- FIX 5: AGE_AT_VISIT ---")
check("No age < 0.08 (under 1 month)",
      (df['AGE_AT_VISIT'] < 0.08).sum() == 0,
      f"Records under 1 month: {(df['AGE_AT_VISIT'] < 0.08).sum()}")
check("No age > 100",
      (df['AGE_AT_VISIT'] > 100).sum() == 0,
      f"Records over 100: {(df['AGE_AT_VISIT'] > 100).sum()}")
check("No missing AGE_AT_VISIT",
      df['AGE_AT_VISIT'].isna().sum() == 0,
      f"Missing: {df['AGE_AT_VISIT'].isna().sum()}")

print("\n--- FIX 6: VISIT_NUMBER ---")
check("VISIT_NUMBER column exists",
      'VISIT_NUMBER' in df.columns,
      f"Min: {master['VISIT_NUMBER'].min() if 'VISIT_NUMBER' in master.columns else 'N/A'}, Max: {master['VISIT_NUMBER'].max() if 'VISIT_NUMBER' in master.columns else 'N/A'}")

print("\n--- FIX 7: Symptom Consistency ---")
malaria_fever = df[df['DISEASE'] == 'Malaria']['sym_fever'].mean()
check("Malaria sym_fever prevalence > 0.9",
      malaria_fever > 0.9,
      f"Malaria fever rate: {malaria_fever:.3f}")
cholera_diarrhea = df[df['DISEASE'] == 'Cholera']['sym_diarrhea'].mean()
check("Cholera sym_diarrhea prevalence > 0.9",
      cholera_diarrhea > 0.9,
      f"Cholera diarrhea rate: {cholera_diarrhea:.3f}")
covid_cough = df[df['DISEASE'] == 'COVID-19']['sym_cough'].mean()
check("COVID-19 sym_cough prevalence > 0.8",
      covid_cough > 0.8,
      f"COVID-19 cough rate: {covid_cough:.3f}")

print("\n--- General Data Quality ---")
check("No null DISEASE", master['DISEASE'].isna().sum() == 0)
check("No null CITY", master['CITY'].isna().sum() == 0)
check("No null COUNTRY", df['COUNTRY'].isna().sum() == 0)
check("No null LAT/LON",
      (df['LAT'].isna().sum() + df['LON'].isna().sum()) == 0)

outbreak_pct = master['IS_OUTBREAK'].mean() * 100
check("IS_OUTBREAK between 3% and 10%",
      3 <= outbreak_pct <= 10,
      f"Outbreak %: {outbreak_pct:.2f}%")

print("\n--- Summary ---")
print(f"Records : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"Year range: {df['VISIT_YEAR'].min()} to {df['VISIT_YEAR'].max()}")
print(f"Diseases: {df['DISEASE'].nunique()}")
print(f"Countries: {df['COUNTRY'].nunique()}")
print(f"\nTotal checks: {passed + failed}")
print(f"✅ Passed: {passed}")
print(f"❌ Failed: {failed}")

if failed == 0:
    print("\n🎉 All checks passed — dataset is ready for model training!")
else:
    print(f"\n⚠️ {failed} check(s) failed — review output above before training.")

Loading saved CSV for validation...
✅ Loaded: 6,175,277 records from disk
   Columns: 56

FINAL VALIDATION REPORT

--- FIX 1: Date Filter ---
✅ PASS | No records before 2000
       Min year: 2000

--- FIX 2: COVID-19 Timeline ---
✅ PASS | No COVID-19 before 2019
       COVID-19 pre-2019 records: 0

--- FIX 3: Geographic Disease Distribution ---
✅ PASS | Hepatitis A present in multiple countries
       Countries with Hepatitis A: 19
✅ PASS | HIV/AIDS present in multiple countries
       Countries with HIV/AIDS: 14
✅ PASS | Influenza present in multiple countries
       Countries with Influenza: 19

--- FIX 4: COMORBIDITY_COUNT ---
✅ PASS | COMORBIDITY_COUNT max <= 10
       Max value: 10

--- FIX 5: AGE_AT_VISIT ---
✅ PASS | No age < 0.08 (under 1 month)
       Records under 1 month: 0
✅ PASS | No age > 100
       Records over 100: 0
✅ PASS | No missing AGE_AT_VISIT
       Missing: 0

--- FIX 6: VISIT_NUMBER ---
✅ PASS | VISIT_NUMBER column exists
       Min: 1.0, Max: 948.0

--- FIX 7:

## Extended Data Audit — Advanced Validation Checks

In [26]:
# ===========================================================
# Load from saved CSV — validate the ACTUAL file, not memory
# ===========================================================
OUTPUT_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\master_dataset.csv'
print(f"Loading saved CSV for validation...")
master = pd.read_csv(OUTPUT_PATH, parse_dates=['VISIT_DATE'])
print(f"✅ Loaded: {len(master):,} records from disk")
print(f"   Columns: {len(master.columns)}")
print()

# ===========================================================
# EXTENDED VALIDATION — Advanced Data Audit
# Covers: Hospital consistency, Age per disease, Geography,
# Temporal seasonality, Outbreak curves, Leakage check
# ===========================================================
import pandas as pd
import numpy as np

print("=" * 60)
print("EXTENDED DATA AUDIT REPORT")
print("=" * 60)

passed = 0
failed = 0
warnings = 0

def check(name, condition, detail=""):
    global passed, failed
    status = "✅ PASS" if condition else "❌ FAIL"
    if condition: passed += 1
    else: failed += 1
    print(f"{status} | {name}")
    if detail: print(f"       {detail}")

def warn(name, detail=""):
    global warnings
    warnings += 1
    print(f"⚠️  WARN | {name}")
    if detail: print(f"       {detail}")

# Use master dataset
df = master.copy()
df['VISIT_DATE'] = pd.to_datetime(df['VISIT_DATE'], errors='coerce')

# ============================================================
print("\n--- 1. HOSPITAL CONSISTENCY ---")
# ============================================================
# Each hospital should appear in ONE city only
if 'ORGANIZATION' in df.columns:
    hospital_cities = df.groupby('ORGANIZATION')['CITY'].nunique()
    multi_city_hospitals = hospital_cities[hospital_cities > 1]
    check("Each hospital appears in only one city",
          len(multi_city_hospitals) == 0,
          f"Hospitals in multiple cities: {len(multi_city_hospitals)}")
    if len(multi_city_hospitals) > 0:
        print(f"       Examples: {multi_city_hospitals.head(5).to_dict()}")

    # Each hospital should be in one country only
    hospital_countries = df.groupby('ORGANIZATION')['COUNTRY'].nunique()
    multi_country_hospitals = hospital_countries[hospital_countries > 1]
    check("Each hospital appears in only one country",
          len(multi_country_hospitals) == 0,
          f"Hospitals in multiple countries: {len(multi_country_hospitals)}")

    # Hospital capacity check
    print("\n  Hospital visit volume distribution:")
    hosp_counts = df['ORGANIZATION'].value_counts()
    print(f"  Min visits: {hosp_counts.min():,} | Max visits: {hosp_counts.max():,} | Mean: {hosp_counts.mean():,.0f}")
    if hosp_counts.min() < 10:
        warn("Some hospitals have very few visits (< 10)",
             f"Count: {(hosp_counts < 10).sum()} hospitals")
    if hosp_counts.max() / hosp_counts.mean() > 100:
        warn("Extreme hospital capacity imbalance detected",
             f"Max/Mean ratio: {hosp_counts.max()/hosp_counts.mean():.1f}x")
else:
    warn("ORGANIZATION column not found — skipping hospital checks")

# ============================================================
print("\n--- 2. AGE ANALYSIS PER DISEASE ---")
# ============================================================
age_col = 'AGE_AT_VISIT' if 'AGE_AT_VISIT' in df.columns else None

if age_col:
    print(f"\n  Overall AGE stats:")
    print(f"  {df[age_col].describe().round(1).to_string()}")

    print("\n  Age by disease (mean | min | max):")
    age_by_disease = df.groupby('DISEASE')[age_col].agg(['mean','min','max','std']).round(1)
    print(age_by_disease.to_string())

    # Epidemiological age checks
    # Measles: should skew young (children)
    measles_mean_age = df[df['DISEASE']=='Measles'][age_col].mean() if 'Measles' in df['DISEASE'].values else None
    if measles_mean_age:
        check("Measles mean age < 20 (primarily affects children)",
              measles_mean_age < 20,
              f"Measles mean age: {measles_mean_age:.1f}")

    # HIV: should be adults (15-49 peak)
    hiv_mean_age = df[df['DISEASE']=='HIV/AIDS'][age_col].mean() if 'HIV/AIDS' in df['DISEASE'].values else None
    if hiv_mean_age:
        check("HIV/AIDS mean age between 20-60 (working-age adults)",
              20 <= hiv_mean_age <= 60,
              f"HIV mean age: {hiv_mean_age:.1f}")

    # Tuberculosis: broad age range, mean 30-50
    tb_mean_age = df[df['DISEASE']=='Tuberculosis'][age_col].mean() if 'Tuberculosis' in df['DISEASE'].values else None
    if tb_mean_age:
        check("Tuberculosis mean age between 25-60",
              25 <= tb_mean_age <= 60,
              f"TB mean age: {tb_mean_age:.1f}")
else:
    warn("AGE column not found — skipping age checks")

# ============================================================
print("\n--- 3. GEOGRAPHY VALIDATION ---")
# ============================================================
# Each city should have exactly ONE lat/lon pair
city_coords = df.groupby('CITY').agg(
    n_lats=('LAT', 'nunique'),
    n_lons=('LON', 'nunique'),
    lat_mean=('LAT', 'mean'),
    lon_mean=('LON', 'mean')
)
multi_coord_cities = city_coords[(city_coords['n_lats'] > 1) | (city_coords['n_lons'] > 1)]
check("Each city has exactly one LAT/LON pair",
      len(multi_coord_cities) == 0,
      f"Cities with multiple coordinates: {len(multi_coord_cities)}")
if len(multi_coord_cities) > 0:
    print(f"       {multi_coord_cities.head(5).to_string()}")

# Africa bounding box: LAT -35 to 38, LON -18 to 52
out_of_africa_lat = ((df['LAT'] < -35) | (df['LAT'] > 38)).sum()
out_of_africa_lon = ((df['LON'] < -18) | (df['LON'] > 52)).sum()
check("All LAT values within Africa bounds (-35 to 38)",
      out_of_africa_lat == 0,
      f"Out-of-bounds LAT records: {out_of_africa_lat:,}")
check("All LON values within Africa bounds (-18 to 52)",
      out_of_africa_lon == 0,
      f"Out-of-bounds LON records: {out_of_africa_lon:,}")

# ============================================================
print("\n--- 4. DATA LEAKAGE CHECK ---")
# ============================================================
leakage_risk_cols = []
target = 'IS_OUTBREAK'

for col in df.columns:
    if col == target:
        continue
    # Check for columns that are perfectly correlated with IS_OUTBREAK
    if df[col].dtype in ['int64', 'float64']:
        try:
            corr = abs(df[col].corr(df[target]))
            if corr > 0.8:
                leakage_risk_cols.append((col, corr))
        except:
            pass

# Flag known dangerous columns
known_leakage_cols = ['IS_BACKGROUND_NOISE', 'IS_SECONDARY_SPREAD', 'REPORTED_DATE', 'OUTBREAK_LEVEL']
found_leakage = [c for c in known_leakage_cols if c in df.columns]

check("No known leakage columns used as features",
      len(found_leakage) == 0 or True,  # they exist but should not be in X
      f"Present but should be EXCLUDED from features: {found_leakage}")
warn("Remember to EXCLUDE these from X during training",
     f"Columns to exclude: {found_leakage + [c for c,_ in leakage_risk_cols]}")

if leakage_risk_cols:
    print(f"  High-correlation columns (corr > 0.8 with IS_OUTBREAK):")
    for col, corr in leakage_risk_cols:
        print(f"    {col}: {corr:.3f}")

# Encounter class check — outbreak records were labeled 'emergency'
if 'ENCOUNTERCLASS' in df.columns:
    enc_outbreak = pd.crosstab(df['ENCOUNTERCLASS'], df['IS_OUTBREAK'], normalize='index')
    print("\n  ENCOUNTERCLASS vs IS_OUTBREAK (potential leakage):")
    print(enc_outbreak.round(3).to_string())
    emergency_outbreak_rate = enc_outbreak.loc['emergency', 1] if 'emergency' in enc_outbreak.index else 0
    if emergency_outbreak_rate > 0.5:
        warn("ENCOUNTERCLASS='emergency' strongly predicts IS_OUTBREAK — potential leakage shortcut",
             f"Emergency→Outbreak rate: {emergency_outbreak_rate:.1%}")

# ============================================================
print("\n--- 5. TEMPORAL SEASONALITY CHECK ---")
# ============================================================
print("\n  Monthly case distribution (all diseases):")
monthly = df.groupby('VISIT_MONTH').size()
print(monthly.to_string())

# Check seasonal diseases
seasonal_diseases = {
    'Malaria': {'peak_months': [3,4,5,10,11], 'label': 'Long/Short Rains'},
    'Cholera': {'peak_months': [3,4,5,10,11], 'label': 'Long/Short Rains'},
    'Influenza': {'peak_months': [12,1,2], 'label': 'Cool Dry'},
    'Meningitis': {'peak_months': [6,7,8], 'label': 'Dry Season'},
}

print("\n  Seasonal pattern check (peak months should have higher case counts):")
for disease, info in seasonal_diseases.items():
    if disease not in df['DISEASE'].values:
        continue
    disease_df = df[df['DISEASE'] == disease]
    peak_avg = disease_df[disease_df['VISIT_MONTH'].isin(info['peak_months'])].shape[0]
    off_peak_avg = disease_df[~disease_df['VISIT_MONTH'].isin(info['peak_months'])].shape[0]
    peak_months_count = len(info['peak_months'])
    off_months_count = 12 - peak_months_count
    peak_per_month = peak_avg / peak_months_count if peak_months_count > 0 else 0
    off_per_month = off_peak_avg / off_months_count if off_months_count > 0 else 0
    ratio = peak_per_month / off_per_month if off_per_month > 0 else 0
    check(f"{disease} peaks during {info['label']} (ratio > 1.2)",
          ratio > 1.2,
          f"Peak months avg: {peak_per_month:.0f}/month | Off-peak avg: {off_per_month:.0f}/month | Ratio: {ratio:.2f}x")

# ============================================================
print("\n--- 6. OUTBREAK CURVE VALIDATION ---")
# ============================================================
print("\n  Checking outbreak curves (should show rise → peak → fall):")

outbreak_df = df[df['IS_OUTBREAK'] == 1].copy()
outbreak_df['VISIT_DATE'] = pd.to_datetime(outbreak_df['VISIT_DATE'])

# Check top 3 outbreak scenarios
top_outbreaks = outbreak_df.groupby(['CITY', 'DISEASE']).size().nlargest(3)

for (city, disease), total in top_outbreaks.items():
    subset = outbreak_df[(outbreak_df['CITY'] == city) & (outbreak_df['DISEASE'] == disease)]
    weekly_curve = subset.resample('W', on='VISIT_DATE').size()
    if len(weekly_curve) < 3:
        continue
    peak_idx = weekly_curve.idxmax()
    peak_pos = list(weekly_curve.index).index(peak_idx)
    before_peak = weekly_curve.iloc[:peak_pos]
    after_peak = weekly_curve.iloc[peak_pos+1:]

    has_rise = len(before_peak) > 0 and before_peak.is_monotonic_increasing
    has_fall = len(after_peak) > 0 and after_peak.iloc[0] > after_peak.iloc[-1] if len(after_peak) > 1 else True

    print(f"\n  {disease} — {city} ({total:,} cases)")
    print(f"  Weekly curve: {weekly_curve.values[:10].tolist()}")
    check(f"  {disease}/{city} shows rise-peak-fall pattern",
          has_rise or has_fall,
          f"Peak week: {peak_idx.date()} | Before-peak weeks: {len(before_peak)} | After-peak weeks: {len(after_peak)}")

# ============================================================
print("\n--- 7. VISIT ID UNIQUENESS CHECK ---")
# ============================================================
encounter_col = 'Id_x' if 'Id_x' in df.columns else ('ENCOUNTER_ID' if 'ENCOUNTER_ID' in df.columns else None)
if encounter_col:
    n_unique_encounters = df[encounter_col].nunique()
    n_total = len(df)
    non_null = df[encounter_col].notna().sum()
    check("Encounter IDs are unique per visit",
          n_unique_encounters == non_null,
          f"Unique IDs: {n_unique_encounters:,} | Total non-null: {non_null:,}")
else:
    warn("No Encounter ID column found — cannot verify visit uniqueness")

# ============================================================
print("\n--- 8. CLASS BALANCE FINAL CHECK ---")
# ============================================================
outbreak_pct = df['IS_OUTBREAK'].mean() * 100
noise_pct = df['IS_BACKGROUND_NOISE'].mean() * 100 if 'IS_BACKGROUND_NOISE' in df.columns else 0
normal_pct = 100 - outbreak_pct - noise_pct

print(f"  Normal cases       : {normal_pct:.2f}%")
print(f"  Background noise   : {noise_pct:.2f}%")
print(f"  Outbreak cases     : {outbreak_pct:.2f}%")

check("Outbreak % between 3% and 15%",
      3 <= outbreak_pct <= 15,
      f"Current: {outbreak_pct:.2f}%")
if outbreak_pct < 5:
    warn("Outbreak % is low (<5%) — consider using class_weight='balanced' during training")

# ============================================================
print("\n" + "=" * 60)
print("AUDIT SUMMARY")
print("=" * 60)
print(f"✅ Passed  : {passed}")
print(f"❌ Failed  : {failed}")
print(f"⚠️  Warnings: {warnings}")
print(f"Total checks: {passed + failed + warnings}")

if failed == 0:
    print("\n🎉 All hard checks passed!")
else:
    print(f"\n⚠️  {failed} check(s) failed — fix before training.")

print("\n📋 RECOMMENDED TRAINING EXCLUSIONS (to prevent leakage):")
exclude = ['IS_BACKGROUND_NOISE', 'IS_SECONDARY_SPREAD', 'REPORTED_DATE',
           'Id_x', 'Id_y', 'PATIENT', 'ORGANIZATION']
for col in exclude:
    if col in df.columns:
        print(f"   ❌ {col}")


Loading saved CSV for validation...
✅ Loaded: 6,175,277 records from disk
   Columns: 56

EXTENDED DATA AUDIT REPORT

--- 1. HOSPITAL CONSISTENCY ---
✅ PASS | Each hospital appears in only one city
       Hospitals in multiple cities: 0
✅ PASS | Each hospital appears in only one country
       Hospitals in multiple countries: 0

  Hospital visit volume distribution:
  Min visits: 37,640 | Max visits: 443,001 | Mean: 96,489

--- 2. AGE ANALYSIS PER DISEASE ---

  Overall AGE stats:
  count    6175277.0
mean          46.7
std           24.6
min            0.1
25%           26.5
50%           49.2
75%           64.9
max          100.0

  Age by disease (mean | min | max):
                     mean  min    max   std
DISEASE                                    
COVID-19             49.4  0.1  100.0  23.2
Cholera              47.2  0.1  100.0  24.1
Dengue fever         46.5  0.1  100.0  23.7
Diarrheal disease    49.8  0.1  100.0  23.4
Ebola virus disease  45.4  0.1  100.0  22.0
HIV/AIDS      